# SQuARE dataset statistics (DBpedia and Wikidata)

Computes the dataset statistics reported in the paper from the final component-disjoint
splits produced by `rebuild_*_clean_splits.py`. Both graphs are processed with identical
definitions.

| Statistic | Paper |
|---|---|
| Queries, anchors and stored pairs per split | Table 1 |
| Positive pairs with identical answer sets ($J=1$) | §3.1, Table 1 |
| Queries with `LIMIT` / `OFFSET` | §3.1 |
| Largest answer set in the test split; recurring answer sizes (row-cap check) | §3.2 |
| Answers containing blank nodes | §3.2 |
| Query templates; test anchors whose template occurs in training | §3.2, Table 1, §4 |
| Nonzero-overlap negatives; projection-arity agreement | §3.2 |
| In-batch candidates that may be unlabeled positives (sampler simulation) | §3.3 |
| Query length in characters and in tokens per encoder | §3.3 |
| Query execution time | §3.2 |

**Location:** place this notebook in the `datasets/` folder. All paths are relative to it:

```
datasets/
├── dataset_statistics.ipynb
├── dbpedia-splits/   dbpedia-{train,valid,test}.json,
│                     query_execution_cache.jsonl, split_report.json, unique_queries.json
└── wikidata-splits/  wikidata-{train,valid,test}.json,
                      query_execution_cache.jsonl, split_report.json, unique_queries.json
```
Results are written to `statistics/dataset_statistics.json`; the notebook only reads the input files.

## 1. Configuration

In [1]:
from pathlib import Path

def dataset_files(name):
    d = Path(f"{name}-splits")
    return {
        "train": d / f"{name}-train.json",
        "valid": d / f"{name}-valid.json",
        "test":  d / f"{name}-test.json",
        "test_rq3":       d / f"{name}-test-rq3.json",
        "cache":          d / "query_execution_cache.jsonl",
        "split_report":   d / "split_report.json",
        "unique_queries": d / "unique_queries.json",
    }

DATASETS = {"dbpedia": dataset_files("dbpedia"), "wikidata": dataset_files("wikidata")}

# Repository root relative to datasets/ (used to read the encoder configurations)
REPO_ROOT = Path("..")

# Sampler simulation: must match the training configuration
SIM_BATCH_SIZE = 4          # anchors per batch (B)
SIM_EXPLICIT_NEGATIVES = 4  # explicit negatives per anchor (m)
SIM_EPOCHS = 5              # independent random epochs (mean and max are reported)
SIM_SEED = 42

# Token lengths per encoder (requires `transformers`; Gemma requires HF_TOKEN)
RUN_TOKENIZERS = True
MODEL_KEYS = ["mpnet", "e5-base-v2", "bge-base", "gte-base", "qwen3-embedding",
              "gemma", "codebert", "graphcodebert", "qwen2.5-coder", "spbert"]

# Keep hashed answer rows of training queries in memory for the exact in-batch check.
# Set to False if memory is limited; the component-based estimate is still reported.
HASH_TRAIN_ANSWERS = True

OUTPUT_FILE = Path("statistics/dataset_statistics.json")

## 2. Helpers

In [2]:
import json, re, hashlib, random, statistics
from collections import Counter
import numpy as np
import pandas as pd

def normalize_query(q: str) -> str:
    """Whitespace normalization, identical to the rebuild script and the training code."""
    return re.sub(r"\s+", " ", q.strip())

def query_hash(q: str) -> str:
    return hashlib.sha256(normalize_query(q).encode("utf-8")).hexdigest()

def load_json(p):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

def exists(p):
    return p is not None and Path(p).exists()

def pct(x, n, d=1):
    return float("nan") if n == 0 else round(100.0 * x / n, d)

RESULTS = {name: {} for name in DATASETS}
def record(ds, key, value, note=""):
    RESULTS[ds][key] = value
    print(f"[{ds:8s}] {key:48s} = {value}" + (f"   ({note})" if note else ""))

## 3. Load the splits

In [3]:
SPLITS = ("train", "valid", "test")
DATA = {}
for ds, cfg in DATASETS.items():
    recs = {}
    for s in SPLITS:
        if not exists(cfg[s]):
            print(f"!! {ds}/{s}: file not found -> {cfg[s]}")
            continue
        recs[s] = load_json(cfg[s])
    DATA[ds] = recs
    print(ds, {s: len(r) for s, r in recs.items()})

def split_queries(records):
    """Unique queries of a split (anchors and stored candidates): normalized -> original."""
    qs = {}
    for r in records:
        for q in [r["anchor_query"]] + [c["query"] for c in r["positive_queries"]] \
                 + [c["query"] for c in r["negative_queries"]]:
            qs.setdefault(normalize_query(q), q)
    return qs

QUERIES = {ds: {s: split_queries(DATA[ds][s]) for s in DATA[ds]} for ds in DATA}
ALLQ = {ds: {k: v for s in QUERIES[ds] for k, v in QUERIES[ds][s].items()} for ds in DATA}

dbpedia {'train': 3737, 'valid': 469, 'test': 468}
wikidata {'train': 6734, 'valid': 842, 'test': 842}


## 4. Split sizes and positive components (Table 1)

In [4]:
for ds in DATA:
    qc = {s: len(QUERIES[ds][s]) for s in QUERIES[ds]}
    ac = {s: len(DATA[ds][s]) for s in DATA[ds]}
    npos = sum(len(r["positive_queries"]) for s in DATA[ds] for r in DATA[ds][s])
    nneg = sum(len(r["negative_queries"]) for s in DATA[ds] for r in DATA[ds][s])
    record(ds, "unique_queries", len(ALLQ[ds]))
    record(ds, "queries_train_valid_test", " / ".join(f"{qc.get(s, 0):,}" for s in SPLITS))
    record(ds, "anchors_train_valid_test", " / ".join(f"{ac.get(s, 0):,}" for s in SPLITS))
    record(ds, "stored_positive_pairs", npos)
    record(ds, "stored_negative_pairs", nneg)
    rep = DATASETS[ds]["split_report"]
    if exists(rep):
        rep = load_json(rep)
        for k in ["input_unique_queries_before_filter", "excluded_scalar_aggregates",
                  "input_unique_queries", "execution_status_counts", "usable_nonempty_queries",
                  "positive_components", "largest_positive_component",
                  "median_positive_component", "non_singleton_components"]:
            if k in rep:
                record(ds, "split_report." + k, rep[k])
        if "endpoint" in rep:
            print(f"   note: split_report.json contains the endpoint URL; "
                  "remove it before the anonymized release.")

[dbpedia ] unique_queries                                   = 4727
[dbpedia ] queries_train_valid_test                         = 3,781 / 473 / 473
[dbpedia ] anchors_train_valid_test                         = 3,737 / 469 / 468
[dbpedia ] stored_positive_pairs                            = 39430
[dbpedia ] stored_negative_pairs                            = 93480
[dbpedia ] split_report.input_unique_queries                = 4727
[dbpedia ] split_report.execution_status_counts             = {'success': 4727}
[dbpedia ] split_report.usable_nonempty_queries             = 4727
[dbpedia ] split_report.positive_components                 = 1156
[dbpedia ] split_report.largest_positive_component          = 161
   note: split_report.json contains the endpoint URL; remove it before the anonymized release.
[wikidata] unique_queries                                   = 9679
[wikidata] queries_train_valid_test                         = 7,743 / 968 / 968
[wikidata] anchors_train_valid_test             

## 5. Label statistics: identical answer sets and nonzero-overlap negatives
Counted over stored directed anchor–candidate pairs across all splits (as in Table 1).

In [5]:
for ds in DATA:
    pos = [c["answer_jaccard"] for s in DATA[ds] for r in DATA[ds][s] for c in r["positive_queries"]]
    neg = [c["answer_jaccard"] for s in DATA[ds] for r in DATA[ds][s] for c in r["negative_queries"]]
    n_j1 = sum(1 for j in pos if j >= 1.0 - 1e-9)
    n_nz = sum(1 for j in neg if j > 0.0)
    record(ds, "positive_pairs_identical_answers_pct", pct(n_j1, len(pos)), f"{n_j1:,}/{len(pos):,}")
    record(ds, "negatives_nonzero_overlap_pct", pct(n_nz, len(neg)), f"{n_nz:,}/{len(neg):,}")

[dbpedia ] positive_pairs_identical_answers_pct             = 78.9   (31,094/39,430)
[dbpedia ] negatives_nonzero_overlap_pct                    = 38.1   (35,578/93,480)
[wikidata] positive_pairs_identical_answers_pct             = 79.4   (29,540/37,208)
[wikidata] negatives_nonzero_overlap_pct                    = 52.5   (88,381/168,360)


## 6. SPARQL tokenizer, templates and projection arity
A **template** is the query with prefixed names expanded, variables renamed in order of
appearance and constants abstracted. Two definitions are computed:

* `so`  – IRIs and literals in **subject and object positions of triple patterns** are
  abstracted (the definition stated in §3.2); constants in `FILTER`, `VALUES` and `BIND` are kept.
* `all` – additionally abstracts constants in `FILTER`, `VALUES`, `BIND` and the projection.

Predicates (including property paths) and `LIMIT`/`OFFSET` values are always kept.
The paper should report one definition, applied to both graphs.

In [6]:
TOKEN_RE = re.compile(r"""
 (?P<ws>\s+)
|(?P<comment>\#[^\n]*)
|(?P<iri><[^<>"{}|^`\\\s]*>)
|(?P<str>"{3}(?:[^"\\]|\\.|"(?!""))*"{3}|'{3}(?:[^'\\]|\\.|'(?!''))*'{3}|"(?:[^"\\\n]|\\.)*"|'(?:[^'\\\n]|\\.)*')
|(?P<langdt>@[A-Za-z]+(?:-[A-Za-z0-9]+)*|\^\^)
|(?P<var>[?$][A-Za-z0-9_·-￿]+)
|(?P<num>[+-]?(?:\d+\.\d*(?:[eE][+-]?\d+)?|\.\d+(?:[eE][+-]?\d+)?|\d+(?:[eE][+-]?\d+)?))
|(?P<pname>(?:[A-Za-z][\w.-]*)?:(?:[\w\-.:%]|\\.)*(?<!\.))
|(?P<bnode>_:[\w.-]+|\[\s*\])
|(?P<kw>[A-Za-z_][A-Za-z0-9_]*)
|(?P<op>\|\||&&|!=|<=|>=|\^|[{}()\[\];,./|*+?!=<>-])
""", re.X)

def tokenize(q):
    out, pos = [], 0
    while pos < len(q):
        m = TOKEN_RE.match(q, pos)
        if not m:
            out.append(("other", q[pos])); pos += 1; continue
        k, v, pos = m.lastgroup, m.group(), m.end()
        if k not in ("ws", "comment"):
            out.append((k, v))
    return out

RDF_TYPE = "<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>"
GROUP_KW = {"OPTIONAL", "UNION", "MINUS", "GRAPH", "SERVICE", "WHERE", "SILENT"}
PATH_OPS = {"/", "|", "*", "+", "?", ")"}

def preprocess_tokens(q):
    """Drop PREFIX/BASE, expand prefixed names, merge literals with language tag or datatype."""
    toks = tokenize(q)
    prefixes, body, i = {}, [], 0
    while i < len(toks):
        k, v = toks[i]
        if k == "kw" and v.upper() == "PREFIX" and i + 2 < len(toks):
            prefixes[toks[i + 1][1].split(":")[0]] = toks[i + 2][1][1:-1]; i += 3; continue
        if k == "kw" and v.upper() == "BASE":
            i += 2; continue
        if k == "pname":
            p, _, local = v.partition(":")
            if p in prefixes:
                k, v = "iri", "<" + prefixes[p] + local.replace("\\", "") + ">"
        if k == "kw" and v == "a":
            k, v = "iri", RDF_TYPE
        body.append((k, v)); i += 1
    out, i = [], 0
    while i < len(body):
        k, v = body[i]
        if k == "str":
            if i + 1 < len(body) and body[i + 1][0] == "langdt":
                if body[i + 1][1] == "^^" and i + 2 < len(body):
                    out.append(("lit", v + "^^" + body[i + 2][1])); i += 3; continue
                out.append(("lit", v + body[i + 1][1])); i += 2; continue
            out.append(("lit", v)); i += 1; continue
        if k == "num" or (k == "kw" and v.lower() in ("true", "false")):
            out.append(("lit", v.lower())); i += 1; continue
        out.append((k, v)); i += 1
    return out

def template(q, mode="so"):
    toks = preprocess_tokens(q)
    out, varmap = [], {}
    state, brace = "S", 0
    expr_mode, expr_paren, values_mode, limit_next = False, 0, False, False
    C = lambda v: "<C>" if mode == "all" else v
    for idx, (k, v) in enumerate(toks):
        if k == "var":
            v = varmap.setdefault(v[1:], f"?v{len(varmap)}")
        U = v.upper() if k == "kw" else None
        if U in ("FILTER", "BIND", "HAVING"):
            out.append(U); expr_mode, expr_paren = True, 0; continue
        if U == "VALUES":
            out.append(U); values_mode = True; continue
        if U in ("LIMIT", "OFFSET"):
            out.append(U); limit_next = True; continue
        if limit_next and k == "lit":
            out.append(v); limit_next = False; continue
        limit_next = False
        if expr_mode:
            expr_paren += (v == "(") - (v == ")")
            out.append(C(v) if k in ("iri", "lit") else (U or v))
            if expr_paren == 0 and v == ")":
                expr_mode, state = False, "S"
            continue
        if values_mode:
            out.append(C(v) if k in ("iri", "lit") else (U or v))
            if v == "}":
                values_mode, state = False, "S"
            continue
        if v == "{":
            brace += 1; state = "S"; out.append(v); continue
        if v == "}":
            brace -= 1; state = "S"; out.append(v); continue
        if U in GROUP_KW:
            out.append(U); state = "S"; continue
        if k == "kw":
            out.append(U); continue
        if brace == 0:                       # projection and solution modifiers
            out.append(C(v) if k in ("iri", "lit") else v); continue
        if v == ".": state = "S"; out.append(v); continue
        if v == ";": state = "P"; out.append(v); continue
        if v == ",": state = "O"; out.append(v); continue
        if v == "[": state = "P"; out.append(v); continue
        if v == "]": state = "O_DONE"; out.append(v); continue
        if state == "S":
            out.append("<S>" if k in ("iri", "lit") else v)
            if k in ("iri", "lit", "var", "bnode"): state = "P"
            continue
        if state == "P":                     # predicate or property path: kept verbatim
            out.append(v)
            nxt = toks[idx + 1][1] if idx + 1 < len(toks) else ""
            if (k in ("iri", "var") or v in (")", "*", "+", "?")) and nxt not in PATH_OPS:
                state = "O"
            continue
        if state == "O":
            out.append("<O>" if k in ("iri", "lit") else v)
            if k in ("iri", "lit", "var", "bnode"): state = "O_DONE"
            continue
        out.append(v)
    return " ".join(out)

def has_limit_offset(q):
    return any(k == "kw" and v.upper() in ("LIMIT", "OFFSET") for k, v in tokenize(q))

def projection_arity_from_text(q):
    """Projected columns of the outer SELECT (SELECT * -> distinct variables in the pattern)."""
    toks = preprocess_tokens(q)
    try:
        i = next(j for j, (k, v) in enumerate(toks) if k == "kw" and v.upper() == "SELECT")
    except StopIteration:
        return None
    j, depth, n, star = i + 1, 0, 0, False
    while j < len(toks):
        k, v = toks[j]
        if depth == 0 and (v == "{" or (k == "kw" and v.upper() in ("WHERE", "FROM"))):
            break
        if v == "(":
            depth += 1
        elif v == ")":
            depth -= 1
        elif v == "*" and depth == 0:
            star = True
        elif k == "var" and depth == 0:
            n += 1
        j += 1
    if star:
        return len({v for k, v in toks[j:] if k == "var"})
    return n

## 7. Load the execution cache

In [7]:
CACHE = {}         # ds -> {query_hash: {"n", "arity", "bnode_rows", "elapsed_s"}}
TRAIN_HASHES = {}  # ds -> {query_hash: sorted unique int64 row hashes} (training queries only)

def load_cache(ds):
    path = DATASETS[ds]["cache"]
    if not exists(path):
        print(f"[{ds}] no execution cache found at {path}")
        return
    wanted = {query_hash(q) for q in ALLQ[ds].values()}
    train_wanted = {query_hash(q) for q in QUERIES[ds].get("train", {}).values()} \
        if HASH_TRAIN_ANSWERS else set()
    info, hashes = {}, {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            h = obj.get("query_hash") or query_hash(obj["query"])
            if h not in wanted or obj.get("status") != "success":
                continue
            answers = obj.get("answers") or []
            info[h] = {"n": obj.get("n_answers", len(answers)),
                       "arity": len(json.loads(answers[0])) if answers else None,
                       "bnode_rows": sum(1 for a in answers if '["bnode"' in a),
                       "elapsed_s": obj.get("elapsed_s")}
            if h in train_wanted:
                hashes[h] = np.unique(np.fromiter((hash(a) for a in answers),
                                                  dtype=np.int64, count=len(answers)))
    CACHE[ds], TRAIN_HASHES[ds] = info, hashes
    missing = len(wanted) - len(info)
    print(f"[{ds}] cache entries matched: {len(info):,} / {len(wanted):,}"
          + (f"   !! {missing} split queries not found in the cache" if missing else ""))

for ds in DATA:
    load_cache(ds)

[dbpedia] cache entries matched: 4,727 / 4,727
[wikidata] cache entries matched: 9,679 / 9,679


## 8. Answer sizes, row-cap check, blank nodes and execution time

In [8]:
for ds in DATA:
    # largest answer set in the test split, from the RQ3 ground-truth file (independent of the cache)
    if exists(DATASETS[ds]["test_rq3"]):
        rq3 = load_json(DATASETS[ds]["test_rq3"])
        n = [q["n_answers"] for q in rq3["queries"]]
        record(ds, "test_rq3.query_count", len(n),
               "must equal the number of test queries; otherwise the file is from an older split")
        record(ds, "max_answer_rows_test[rq3]", max(n), f"{max(n) / 1e6:.2f} million")

    if not CACHE.get(ds):
        continue
    test_sizes = [CACHE[ds][query_hash(q)]["n"] for q in QUERIES[ds]["test"].values()
                  if query_hash(q) in CACHE[ds]]
    all_sizes = [x["n"] for x in CACHE[ds].values()]
    record(ds, "max_answer_rows_test[cache]", max(test_sizes), f"{max(test_sizes) / 1e6:.2f} million")
    record(ds, "max_answer_rows_all", max(all_sizes), f"{max(all_sizes) / 1e6:.2f} million")
    record(ds, "median_answer_rows_all", int(statistics.median(all_sizes)))
    record(ds, "single_row_answers_pct", pct(sum(n == 1 for n in all_sizes), len(all_sizes)))
    record(ds, "most_frequent_sizes_ge_1000", Counter(n for n in all_sizes if n >= 1000).most_common(5),
           "a row cap would appear as one size repeated many times")

    bq = {h for h, x in CACHE[ds].items() if x["bnode_rows"] > 0}
    record(ds, "queries_with_blank_node_answers", len(bq),
           f"{pct(len(bq), len(CACHE[ds]), 2)}% of queries; "
           f"{sum(CACHE[ds][h]['bnode_rows'] for h in bq):,} rows")
    n_pairs = n_bpairs = 0
    for s in DATA[ds]:
        for r in DATA[ds][s]:
            ha = query_hash(r["anchor_query"])
            for c in r["positive_queries"]:
                n_pairs += 1
                n_bpairs += (ha in bq) or (query_hash(c["query"]) in bq)
    record(ds, "positive_pairs_with_blank_nodes_pct", pct(n_bpairs, n_pairs, 2), f"{n_bpairs}/{n_pairs}")

    el = [x["elapsed_s"] for x in CACHE[ds].values() if x["elapsed_s"] is not None]
    if el:
        record(ds, "execution_time_total_hours", round(sum(el) / 3600, 2),
               "sum of per-query times (sequential); divide by the number of workers for wall clock")
        record(ds, "execution_time_median_s", round(statistics.median(el), 2))
        record(ds, "execution_time_p95_s", round(sorted(el)[int(0.95 * len(el))], 2))

[dbpedia ] test_rq3.query_count                             = 473   (must equal the number of test queries; otherwise the file is from an older split)
[dbpedia ] max_answer_rows_test[rq3]                        = 2798774   (2.80 million)
[dbpedia ] max_answer_rows_test[cache]                      = 2798774   (2.80 million)
[dbpedia ] max_answer_rows_all                              = 2798774   (2.80 million)
[dbpedia ] median_answer_rows_all                           = 2
[dbpedia ] single_row_answers_pct                           = 40.6
[dbpedia ] most_frequent_sizes_ge_1000                      = [(2798774, 3), (1058, 3), (1474, 2), (77163, 2), (2143, 2)]   (a row cap would appear as one size repeated many times)
[dbpedia ] queries_with_blank_node_answers                  = 0   (0.0% of queries; 0 rows)
[dbpedia ] positive_pairs_with_blank_nodes_pct              = 0.0   (0/39430)
[dbpedia ] execution_time_total_hours                       = 0.3   (sum of per-query times (sequential); 

## 9. Query templates and template overlap between training and test

In [9]:
TEMPLATES = {}
for ds in DATA:
    for mode in ("so", "all"):
        tmpl = {nq: template(q, mode) for nq, q in ALLQ[ds].items()}
        TEMPLATES[(ds, mode)] = tmpl
        train_t = {tmpl[nq] for nq in QUERIES[ds]["train"]}
        test_anchors = [normalize_query(r["anchor_query"]) for r in DATA[ds]["test"]]
        seen = sum(tmpl[a] in train_t for a in test_anchors)
        test_q = list(QUERIES[ds]["test"])
        seen_q = sum(tmpl[a] in train_t for a in test_q)
        largest = Counter(tmpl.values()).most_common(1)[0][1]
        record(ds, f"templates[{mode}]", len(set(tmpl.values())))
        record(ds, f"test_anchors_with_training_template_pct[{mode}]", pct(seen, len(test_anchors)),
               f"{seen}/{len(test_anchors)}; unseen anchors = {len(test_anchors) - seen}")
        record(ds, f"test_queries_with_training_template_pct[{mode}]", pct(seen_q, len(test_q)))
        record(ds, f"largest_template_share_pct[{mode}]", pct(largest, len(tmpl)))

        # test anchor -> seen / unseen, used for the seen/unseen-template retrieval analysis
        groups = {a: ("seen" if tmpl[a] in train_t else "unseen") for a in test_anchors}
        Path(f"statistics/{ds}").mkdir(parents=True, exist_ok=True)
        Path(f"statistics/{ds}/test_anchor_template_groups_{mode}.json").write_text(
            json.dumps(groups, ensure_ascii=False))

[dbpedia ] templates[so]                                    = 143
[dbpedia ] test_anchors_with_training_template_pct[so]      = 97.9   (458/468; unseen anchors = 10)
[dbpedia ] test_queries_with_training_template_pct[so]      = 97.5
[dbpedia ] largest_template_share_pct[so]                   = 34.7
[dbpedia ] templates[all]                                   = 115
[dbpedia ] test_anchors_with_training_template_pct[all]     = 97.9   (458/468; unseen anchors = 10)
[dbpedia ] test_queries_with_training_template_pct[all]     = 97.5
[dbpedia ] largest_template_share_pct[all]                  = 34.7
[wikidata] templates[so]                                    = 5363
[wikidata] test_anchors_with_training_template_pct[so]      = 46.6   (392/842; unseen anchors = 450)
[wikidata] test_queries_with_training_template_pct[so]      = 44.2
[wikidata] largest_template_share_pct[so]                   = 3.8
[wikidata] templates[all]                                   = 3056
[wikidata] test_anchors_with_tra

## 10. `LIMIT` / `OFFSET` and projection arity

In [10]:
def arity_of(ds, q):
    if CACHE.get(ds):
        a = CACHE[ds].get(query_hash(q), {}).get("arity")
        if a is not None:
            return a
    return projection_arity_from_text(q)

for ds in DATA:
    qs = list(ALLQ[ds].values())
    n = sum(has_limit_offset(q) for q in qs)
    record(ds, "queries_with_limit_offset_pct", pct(n, len(qs), 2), f"{n}/{len(qs)}")

    src = "answer columns" if CACHE.get(ds) else "parsed query text"
    neg_diff = neg_tot = pos_same = pos_tot = 0
    for s in DATA[ds]:
        for r in DATA[ds][s]:
            a = arity_of(ds, r["anchor_query"])
            for c in r["negative_queries"]:
                neg_tot += 1; neg_diff += arity_of(ds, c["query"]) != a
            for c in r["positive_queries"]:
                pos_tot += 1; pos_same += arity_of(ds, c["query"]) == a
    record(ds, "negatives_different_arity_pct", pct(neg_diff, neg_tot), src)
    record(ds, "positives_same_arity_pct", pct(pos_same, pos_tot, 2), src)

[dbpedia ] queries_with_limit_offset_pct                    = 0.0   (0/4727)
[dbpedia ] negatives_different_arity_pct                    = 25.0   (answer columns)
[dbpedia ] positives_same_arity_pct                         = 100.0   (answer columns)
[wikidata] queries_with_limit_offset_pct                    = 0.89   (86/9679)
[wikidata] negatives_different_arity_pct                    = 33.9   (answer columns)
[wikidata] positives_same_arity_pct                         = 100.0   (answer columns)


## 11. Query length (characters and tokens per encoder)

In [11]:
for ds in DATA:
    lens = np.array([len(q) for q in ALLQ[ds].values()])
    record(ds, "max_query_chars", int(lens.max()))
    record(ds, "queries_over_1000_chars_pct", pct(int((lens > 1000).sum()), len(lens), 2),
           f"{int((lens > 1000).sum())}/{len(lens)}")

def model_specs():
    """model key -> (Hugging Face id, prefix, max_length), read from the repository configs."""
    import sys
    specs = {}
    try:
        sys.path.insert(0, str(REPO_ROOT / "src"))
        from result_aware.config import load_model_config
        for key in MODEL_KEYS:
            cfg = load_model_config(key, REPO_ROOT / "configs")
            specs[key] = (cfg["model_name_or_path"], cfg.get("prefix", ""), int(cfg.get("max_length", 512)))
    except Exception as e:
        print("Could not read the encoder configurations:", e)
    return specs

if RUN_TOKENIZERS:
    from transformers import AutoTokenizer
    rows = []
    for key, (hf, prefix, maxlen) in model_specs().items():
        try:
            tok = AutoTokenizer.from_pretrained(hf, trust_remote_code=True)
        except Exception as e:
            print(f"skip {key}: {e}"); continue
        for ds in DATA:
            qs = [prefix + q for q in ALLQ[ds].values()]
            n_tok = [len(x) for x in tok(qs, add_special_tokens=True, truncation=False)["input_ids"]]
            over = sum(n > maxlen for n in n_tok)
            rows.append({"model": key, "dataset": ds, "max_length": maxlen,
                         "over_max_pct": pct(over, len(qs), 2), "n_over": over,
                         "max_tokens": max(n_tok)})
    TOKENS_DF = pd.DataFrame(rows)
    print(TOKENS_DF.to_string(index=False))
    for ds in DATA:
        sub = TOKENS_DF[TOKENS_DF.dataset == ds]
        if len(sub):
            record(ds, "queries_over_max_length_pct_range",
                   f"{sub.over_max_pct.min()}–{sub.over_max_pct.max()}", "min–max over encoders")
            RESULTS[ds]["queries_over_max_length_pct_by_model"] = dict(zip(sub.model, sub.over_max_pct))
            RESULTS[ds]["max_length_by_model"] = dict(zip(sub.model, sub.max_length.astype(int)))

[dbpedia ] max_query_chars                                  = 1145
[dbpedia ] queries_over_1000_chars_pct                      = 3.0   (142/4727)
[wikidata] max_query_chars                                  = 1669
[wikidata] queries_over_1000_chars_pct                      = 4.91   (475/9679)


/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/ganesh-jupyterhub/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Token indices sequence length is longer than the specified maximum sequence length for this model (604 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (606 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (604 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (604 > 512). Running this sequence through the model w

          model  dataset  max_length  over_max_pct  n_over  max_tokens
          mpnet  dbpedia         512          0.00       0         376
          mpnet wikidata         512          0.40      39         787
     e5-base-v2  dbpedia         512          0.00       0         378
     e5-base-v2 wikidata         512          0.41      40         789
       bge-base  dbpedia         512          0.00       0         376
       bge-base wikidata         512          0.40      39         787
       gte-base  dbpedia         512          0.00       0         376
       gte-base wikidata         512          0.40      39         787
qwen3-embedding  dbpedia         512          0.00       0         383
qwen3-embedding wikidata         512          0.75      73         974
          gemma  dbpedia         512          0.00       0         415
          gemma wikidata         512          0.93      90         982
       codebert  dbpedia         512          2.96     140         598
      

## 12. In-batch candidates that may be unlabeled positives
Simulates the training sampler on the training split: anchors are shuffled into batches of
`SIM_BATCH_SIZE`; each anchor contributes one random stored positive and
`SIM_EXPLICIT_NEGATIVES` random stored negatives. For each anchor, the candidates contributed
by the other anchors are inspected after masking the anchor's stored positives.

* **Component estimate** (as in the paper): share of these candidates in the anchor's positive
  component, built from the stored positive pairs.
* **Exact** (requires the cache): share whose true answer Jaccard with the anchor is
  $\geq 0.8$ (unlabeled positives) or in $(0.1, 0.8)$ (intermediate overlap).

In [12]:
class DSU:
    def __init__(self): self.p = {}
    def find(self, x):
        self.p.setdefault(x, x)
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def union(self, a, b): self.p[self.find(a)] = self.find(b)

def exact_jaccard(a, b):
    if len(a) > len(b): a, b = b, a
    if len(a) == 0: return 0.0
    idx = np.searchsorted(b, a)
    idx[idx == len(b)] = 0
    inter = int((b[idx] == a).sum())
    return inter / (len(a) + len(b) - inter)

def simulate(ds):
    recs = DATA[ds]["train"]
    dsu = DSU()
    for s in DATA[ds]:
        for r in DATA[ds][s]:
            a = normalize_query(r["anchor_query"])
            for c in r["positive_queries"]:
                dsu.union(a, normalize_query(c["query"]))
    hashes = TRAIN_HASHES.get(ds) or {}
    exact = len(hashes) > 0
    jcache, per_epoch = {}, []
    for ep in range(SIM_EPOCHS):
        rng = random.Random(SIM_SEED + ep)
        order = list(range(len(recs))); rng.shuffle(order)
        tot = comp = unl = inter = 0
        for b0 in range(0, len(order), SIM_BATCH_SIZE):
            batch = order[b0:b0 + SIM_BATCH_SIZE]
            contrib = {}
            for i in batch:
                r = recs[i]
                pos = rng.choice(r["positive_queries"])["query"]
                negs = rng.sample([c["query"] for c in r["negative_queries"]],
                                  min(SIM_EXPLICIT_NEGATIVES, len(r["negative_queries"])))
                contrib[i] = [pos] + negs
            for i in batch:
                a = recs[i]["anchor_query"]; na = normalize_query(a)
                masked = {normalize_query(c["query"]) for c in recs[i]["positive_queries"]}
                ra, ha = dsu.find(na), query_hash(a)
                for j in batch:
                    if j == i: continue
                    for q in contrib[j]:
                        nq = normalize_query(q)
                        if nq in masked or nq == na: continue
                        tot += 1
                        comp += dsu.find(nq) == ra
                        if exact:
                            hq = query_hash(q)
                            key = (ha, hq) if ha < hq else (hq, ha)
                            if key not in jcache:
                                A, B = hashes.get(ha), hashes.get(hq)
                                if A is None or B is None:
                                    jcache[key] = None
                                else:
                                    lo, hi = sorted((len(A), len(B)))
                                    # J <= |smaller| / |larger|, so J <= 0.1 is guaranteed below this ratio
                                    jcache[key] = 0.0 if hi == 0 or lo / hi <= 0.1 else exact_jaccard(A, B)
                            jv = jcache[key]
                            if jv is not None:
                                unl += jv >= 0.8
                                inter += 0.1 < jv < 0.8
        per_epoch.append((tot, comp, unl, inter))
    tot = np.array([x[0] for x in per_epoch])
    def share(col):
        v = np.array([x[col] for x in per_epoch]) / tot * 100
        return (round(float(v.mean()), 3), round(float(v.max()), 3))
    record(ds, "inbatch_candidates_per_epoch", int(tot.mean()))
    record(ds, "inbatch_same_component_pct(mean,max)", share(1))
    if exact:
        record(ds, "inbatch_unlabeled_positive_pct(mean,max)", share(2))
        record(ds, "inbatch_intermediate_overlap_pct(mean,max)", share(3))

for ds in DATA:
    simulate(ds)

[dbpedia ] inbatch_candidates_per_epoch                     = 55853
[dbpedia ] inbatch_same_component_pct(mean,max)             = (0.455, 0.535)
[dbpedia ] inbatch_unlabeled_positive_pct(mean,max)         = (0.135, 0.163)
[dbpedia ] inbatch_intermediate_overlap_pct(mean,max)       = (3.204, 3.356)
[wikidata] inbatch_candidates_per_epoch                     = 100920
[wikidata] inbatch_same_component_pct(mean,max)             = (0.02, 0.026)
[wikidata] inbatch_unlabeled_positive_pct(mean,max)         = (0.002, 0.003)
[wikidata] inbatch_intermediate_overlap_pct(mean,max)       = (0.61, 0.685)


## 13. Summary and comparison

In [14]:
# Summary table
def fmt(v):
    if isinstance(v, bool):
        return str(v)
    if isinstance(v, int):
        return f"{v:,}"
    if isinstance(v, float):
        return f"{v:,.2f}".rstrip("0").rstrip(".")
    if isinstance(v, (tuple, list)) and all(isinstance(x, (int, float)) for x in v):
        return " / ".join(fmt(x) for x in v)
    return str(v)

# Expand per-encoder dictionaries into one row per encoder
rows = []
for ds, res in RESULTS.items():
    for k, v in res.items():
        if isinstance(v, dict):
            for sub, val in v.items():
                rows.append({"dataset": ds, "statistic": f"{k} [{sub}]", "value": fmt(val)})
        else:
            rows.append({"dataset": ds, "statistic": k, "value": fmt(v)})

SECTIONS = [
    ("Split sizes",        ("unique_queries", "queries_", "anchors_", "stored_", "split_report.")),
    ("Labels",             ("positive_pairs_identical", "negatives_nonzero")),
    ("Answers",            ("max_answer", "median_answer", "single_row", "most_frequent", "test_rq3",
                            "queries_with_blank", "positive_pairs_with_blank")),
    ("Execution time",     ("execution_time",)),
    ("Templates",          ("templates", "test_anchors_with", "test_queries_with", "largest_template")),
    ("Query form",         ("queries_with_limit", "negatives_different_arity", "positives_same_arity")),
    ("Query length",       ("max_query_chars", "queries_over_1000", "queries_over_max_length", "max_length_by_model")),
    ("In-batch candidates", ("inbatch_",)),
]
def section_of(stat):
    for i, (_, prefixes) in enumerate(SECTIONS):
        if stat.startswith(prefixes):
            return i
    return len(SECTIONS)

SUMMARY = (pd.DataFrame(rows)
           .pivot(index="statistic", columns="dataset", values="value")
           .reindex(columns=list(DATASETS))
           .fillna("–"))
SUMMARY["section"] = [(SECTIONS + [("Other", ())])[section_of(s)][0] for s in SUMMARY.index]
SUMMARY["order"] = [section_of(s) for s in SUMMARY.index]
SUMMARY = (SUMMARY.sort_values(["order", "statistic"])
           .drop(columns="order")
           .reset_index()
           .set_index(["section", "statistic"]))

pd.set_option("display.max_rows", 500, "display.max_colwidth", 80, "display.width", 200)
try:
    display(SUMMARY)                  # rendered table in Jupyter
except NameError:
    print(SUMMARY.to_string())        # plain text outside Jupyter

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE.write_text(json.dumps(RESULTS, indent=2, ensure_ascii=False, default=str))
SUMMARY.to_csv(OUTPUT_FILE.with_suffix(".csv"))
print("Saved", OUTPUT_FILE, "and", OUTPUT_FILE.with_suffix(".csv"))

dataset                                                                                                                         dbpedia  \
section             statistic                                                                                                             
Split sizes         anchors_train_valid_test                                                                          3,737 / 469 / 468   
                    queries_over_1000_chars_pct                                                                                       3   
                    queries_over_max_length_pct_by_model [bge-base]                                                                   0   
                    queries_over_max_length_pct_by_model [codebert]                                                                2.96   
                    queries_over_max_length_pct_by_model [e5-base-v2]                                                                 0   
                    queries_over_max_length_pct_by_model [gemma]                                                                      0   
                    queries_over_max_length_pct_by_model [graphcodebert]                                                           2.96   
                    queries_over_max_length_pct_by_model [gte-base]                                                                   0   
                    queries_over_max_length_pct_by_model [mpnet]                                                                      0   
                    queries_over_max_length_pct_by_model [qwen2.5-coder]                                                              0   
                    queries_over_max_length_pct_by_model [qwen3-embedding]                                                            0   
                    queries_over_max_length_pct_by_model [spbert]                                                                     0   
                    queries_over_max_length_pct_range                                                                          0.0–2.96   
                    queries_train_valid_test                                                                          3,781 / 473 / 473   
                    queries_with_blank_node_answers                                                                                   0   
                    queries_with_limit_offset_pct                                                                                     0   
                    split_report.excluded_scalar_aggregates                                                                           –   
                    split_report.execution_status_counts [error]                                                                      –   
                    split_report.execution_status_counts [success]                                                                4,727   
                    split_report.input_unique_queries                                                                             4,727   
                    split_report.input_unique_queries_before_filter                                                                   –   
                    split_report.largest_positive_component                                                                         161   
                    split_report.median_positive_component                                                                            –   
                    split_report.non_singleton_components                                                                             –   
                    split_report.positive_components                                                                              1,156   
                    split_report.usable_nonempty_queries                                                                          4,727   
                    stored_negative_pairs                                                                   

Saved dataset_statistics.json and dataset_statistics.csv
